# CS 3892 / 5892 — Session 16 · Symbolic and bounded model checking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-10-20-symbolic-and-bounded-model-checking.ipynb)

**Tuesday, October 20, 2026.** How a model checker decides a safety property. **Symbolic** model
checking stores sets of states as BDDs and computes the reachable set as a fixpoint. **Bounded**
model checking asks a solver about runs of at most *k* steps — and a clean answer says nothing
about longer runs. **k-induction** turns bounded queries into a proof for runs of every length.

Two machines from session 9: the running counter, and the add-2 counter whose property is true but
not inductive. Every file asserts its own results. NuSMV (free, LGPL) is fetched by the setup cell.

## Setup

Run these two once.

In [ ]:
# --- Setup: find the repo (clone on Colab) and define helpers ---------------
import os, re, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-10-20-symbolic-and-bounded-model-checking"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

SMV = ROOT / "sessions" / SESSION / "smv"
PYD = ROOT / "sessions" / SESSION / "python"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    show(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True)
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

print("ready — helpers: show(path), run(path)")

In [ ]:
# --- NuSMV: fetch the official FBK build if it is not already installed ------
# NuSMV is free (LGPL) from nusmv.fbk.eu. (Same cell as sessions 9-15.)
# 2.7.1 is tried first; it needs glibc >= 2.38 and libedit, which an older
# Colab image may lack, so the static 2.6.0 build is the fallback. Same verdicts.
import shutil

BUILDS = [("2.7.1", "https://nusmv.fbk.eu/distrib/2.7.1/NuSMV-2.7.1-linux64.tar.xz", "J"),
          ("2.6.0", "https://nusmv.fbk.eu/distrib/NuSMV-2.6.0-linux64.tar.gz",       "z")]

def _works(exe):
    """True only if this binary actually model-checks a file -- a missing
    library makes it exit 127 with an error that still contains "NuSMV"."""
    try:
        r = subprocess.run([exe, str(SMV / "06_counter_bmc.smv")],
                           capture_output=True, text=True, timeout=60)
        return "-- specification" in r.stdout
    except Exception:
        return False

NUSMV = shutil.which("NuSMV")
if not (NUSMV and _works(NUSMV)):
    NUSMV = None
    if os.geteuid() == 0 and shutil.which("apt-get"):      # Colab runs as root
        subprocess.run("apt-get install -y -qq libedit2 >/dev/null 2>&1", shell=True)
    for ver, url, z in BUILDS:
        dest = pathlib.Path.home() / ".local" / f"nusmv-{ver}"
        dest.mkdir(parents=True, exist_ok=True)
        subprocess.run(f"curl -sSL {url} | tar -x{z} -C {dest} --strip-components=1",
                       shell=True, check=True)
        if _works(str(dest / "bin" / "NuSMV")):
            NUSMV = str(dest / "bin" / "NuSMV")
            break
        print(f"NuSMV {ver} will not start on this machine -- trying the next build")
assert NUSMV, "could not install NuSMV"
print("NuSMV:", NUSMV)

def smv(path):
    """Run NuSMV on a file in batch mode; return its output minus the banner."""
    r = subprocess.run([NUSMV, str(path)], capture_output=True, text=True)
    return "\n".join(l for l in (r.stdout + r.stderr).splitlines() if not l.startswith("***"))

def results(out):
    """Split NuSMV output into {printed formula: (verdict, counterexample text)}.
    NuSMV does not print verdicts in file order, so match them by formula."""
    res, key = {}, None
    for line in out.splitlines():
        m = re.match(r"-- (?:specification|invariant)\s+(.*?)\s+is (true|false)$", line)
        if m:
            key = re.sub(r"\s+", " ", m.group(1)).strip()
            res[key] = [m.group(2), ""]
        elif key and res[key][0] == "false":
            res[key][1] += line + "\n"
    return {k: tuple(v) for k, v in res.items()}

def verdicts(path):
    """Print one line per property, and return {formula: verdict}."""
    out = smv(path)
    res = results(out)
    for k, (v, _) in res.items():
        print(f"{v:6} {k}")
    return {k: v for k, (v, _) in res.items()}

print("ready -- smv(path), results(out), verdicts(path)")

In [ ]:
# Two more helpers for today: NuSMV in bounded mode, and NuSMV driven by a command script.
import tempfile

def smv_bmc(path, k):
    r = subprocess.run([NUSMV, "-bmc", "-bmc_length", str(k), str(path)], capture_output=True, text=True)
    return [l for l in r.stdout.splitlines() if l.startswith("-- ") and ("counterexample" in l or "specification" in l)]

def smv_commands(path, commands):
    with tempfile.NamedTemporaryFile("w", suffix=".cmd", delete=False) as fh:
        fh.write("\n".join(commands + ["quit"]) + "\n")
    r = subprocess.run([NUSMV, "-source", fh.name, str(path)], capture_output=True, text=True,
                       stdin=subprocess.DEVNULL)
    return [l for l in r.stdout.splitlines() if l.startswith("-- ")]

print("ready -- smv_bmc(path, k), smv_commands(path, [commands])")

## 1. A set of states is a Boolean function

`bdd.py` is a complete reduced ordered BDD in about eighty lines: a node is *(variable, low, high)*,
and two rules — never test a variable whose two branches agree, never build the same node twice —
make every Boolean function's diagram unique.

In [ ]:
show(PYD / "bdd.py")
out = run(PYD / "01_bdd_sets_of_states.py")
assert "7 nodes" in out and "the same node" in out

## 2. Reachability on sets, not states

The counter, and the fixpoint from last Tuesday's slide 6 — with every set a BDD and `Post` computed
on the whole set at once.

In [ ]:
from IPython.display import SVG, display
FIG = ROOT / "sessions" / SESSION / "figures"
display(SVG(filename=str(FIG / "counter.svg")))
out = run(PYD / "02_symbolic_reachability.py")
assert "fixpoint" in out and "EMPTY -> holds" in out

## 3. Bounded model checking: what a clean answer at bound *k* shows

First with Z3, then with NuSMV's own `-bmc` mode on the same machine.

In [ ]:
out = run(PYD / "03_bmc_what_k_shows.py")
assert "COUNTEREXAMPLE  0 -> 1" in out

In [ ]:
f = SMV / "06_counter_bmc.smv"
show(f)
at9, at10 = smv_bmc(f, 9), smv_bmc(f, 10)
print("bound 9: ", [l for l in at9 if "specification" in l] or "no counterexample for either property")
print("bound 10:", [l for l in at10 if "specification" in l])
assert not any("is false" in l for l in at9)
assert any("G x <= 9" in l and "is false" in l for l in at10)
assert not any("G x <= 10" in l and "is false" in l for l in at10)
print()
print("and with BDDs, no bound at all:")
r = verdicts(f)
assert list(r.values()) == ["false", "true"]

## 4. k-induction: a proof for every length

Session 9's add-2 counter. `x != 5` is true and not inductive. No *k* rescues it over the integers;
strengthening does. On a finite range, *k* = 8 does.

In [ ]:
display(SVG(filename=str(FIG / "add2.svg")))
out = run(PYD / "04_k_induction.py")
assert "k = 8   step holds -> PROVED" in out

In [ ]:
display(SVG(filename=str(FIG / "add2_mod16.svg")))
f = SMV / "05_add2_finite.smv"
show(f)
print("BDDs:", verdicts(f))
k7 = smv_commands(f, ["go_bmc", "check_invar_bmc -a een-sorensson -k 7"])
k8 = smv_commands(f, ["go_bmc", "check_invar_bmc -a een-sorensson -k 8"])
print("k-induction, k = 7:", k7[-1])
print("k-induction, k = 8:", k8[-1])
assert "cannot prove" in k7[-1] and "is true" in k8[-1]

## 5. From Thursday: a policy over the agent's history, run for real

Thursday ended on five rules for a coding agent. A gateway that refuses calls can enforce three:

1. never push to `main` — one request, so plain **Cedar**;
2. before a push, the tests must have passed since the last edit — a bounded history: **`since`**;
3. at most three force-pushes an hour — a bounded history: **`count`**.

(4, *every pull request is eventually reviewed*, is liveness; 5, *never send a secret it has read
to the network*, is about pairs of runs. Refusing calls cannot enforce either.)

Session 15's `02_history_monitor.py` was a toy of the operator definitions. This section runs
**Dogwood's own reference interpreter** (v1.1; its authors say it is not for production use) on
the policy files in `dogwood/`. `dogwood_cli.py` finds the `dogwood` command or downloads the
course's unmodified Linux build of it (a few seconds; compiling it on Colab takes minutes).

In [ ]:
DOG = ROOT / "sessions" / SESSION / "dogwood"
show(DOG / "coding_agent.dw")
print("first line of trace_tests.log:")
print((DOG / "trace_tests.log").read_text().splitlines()[0])

The machine that rule 2 needs: one bit and one clock. Cedar never sees it — Dogwood keeps it, and
hands Cedar a single Boolean per temporal condition (part D below).

In [ ]:
display(SVG(filename=str(FIG / "push_monitor.svg")))
out = run(PYD / "07_dogwood_coding_agent.py")
assert "The two versions differ at one request, @720." in out
assert "when { context.policy_1__temporal_0 }" in out

## Try it yourself

- In `01`, change the register width from 4 to 6 bits. Predict both sizes before you run it.
- In `02`, change the counter's cap from 10 to 20 (and widen `x`). How many iterations to the
  fixpoint? Does the diagram for the reachable set get bigger?
- In `03`, find the smallest bound at which BMC disproves `x <= 15` on the counter. Then explain
  the answer.
- In `05`, change the wrap from 16 to 32. Which *k* proves `x != 5` now?
- In `dogwood/coding_agent.dw`, change rule 3's window from `1h` to `2h`. Predict the verdicts at
  @3750 and @3760 before you rerun `07`.
- Write rule 2 with `formerly` instead of `since` ("a passing test run in the last hour"). Which push
  in `trace_tests.log` does it get wrong?